In [19]:
import os, warnings
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [20]:
warnings.filterwarnings("ignore")

In [21]:
ENGINE_URL = ('mysql+pymysql://root:@localhost:3306/bolsa_familia')
engine = create_engine(ENGINE_URL, echo=False)

query = '''
        SELECT `MÊS COMPETÊNCIA`, `UF`, COUNT(*) AS qtd_parcelas, AVG(`VALOR PARCELA`) AS valor_medio, SUM(`VALOR PARCELA`) AS valor_total
        FROM bolsa_familia
        GROUP BY `MÊS COMPETÊNCIA`, `UF`
        '''
df = pd.read_sql(query, engine)


In [22]:
df_uf = df.groupby('UF').agg(media_valor = ('valor_medio', 'mean'), total_parcelas = ('qtd_parcelas', 'sum')).reset_index()

In [26]:
serie = df_uf['media_valor']
q1, q2, q3 = np.percentile(serie, [25, 50, 75])
iqr = q3 - q1
print(f'Media: {serie.mean():.2f} // Mediana: {serie.median():.2f}')
print(f'Q1: {q1:.2f}')
print(f'Q2: {q2:.2f}')
print(f'Q3: {q3:.2f}')
print(f'IQR: {iqr:.2f}')
print(f'Assimetria: {serie.skew():.3f}')
print(f'Curtose: {serie.kurtosis():.3f}')

Media: 675.99 // Mediana: 665.70
Q1: 659.66
Q2: 665.70
Q3: 681.74
IQR: 22.07
Assimetria: 1.362
Curtose: 0.870


In [27]:
df_uf

,UF,media_valor,total_parcelas
0,AC,716.495670,396041
1,AL,676.284181,1604478
2,AM,725.013274,1932927
3,AP,715.441875,366950
4,BA,658.331147,7403292
5,CE,659.898907,4361024
6,DF,665.701533,517548
7,ES,661.125371,921678
8,GO,663.684937,1465255
9,MA,693.613220,3690507


In [32]:
features = df_uf[['media_valor', 'total_parcelas']].values
feature_norm = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_uf['cluster'] = kmeans.fit_predict(feature_norm)
print(df_uf.groupby('cluster')[['UF', 'media_valor']])

NameError: name 'scaler' is not defined